# Philippine Law Assistant — Google Colab (GPU)

Run the **same RAG + law tutor prompt** as the project backend, but with **GPU inference** (usually much faster than on-device CPU).

## Before you run
1. **Runtime → Change runtime type → Hardware accelerator → GPU** (T4 or better).
2. **Get the backend code** into Colab — pick one:
   - **Git clone** your fork (set `REPO_URL` in the next section), or
   - **Upload** the `philippine-law-assistant` folder (zip) to `/content/` and set `BACKEND_ROOT` manually.

## Model
Default: **`Qwen/Qwen2.5-1.5B-Instruct`** (no Llama license gate). Change `MODEL_ID` for another small instruct model.

## 1) Install dependencies

In [ ]:
%pip install -q torch transformers accelerate sentencepiece protobuf safetensors

In [ ]:
import torch

assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime type → GPU"
print("GPU:", torch.cuda.get_device_name(0))

## 2) Point Python at `backend/` (RAG + prompts)

- **Monorepo** (repo contains `philippine-law-assistant/backend/`): clone URL → auto-detect.
- **Standalone** repo (root is `backend/`’s parent): also auto-detected.
- Or **comment out** the clone block and set `BACKEND_ROOT` after uploading a zip.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get(
    "PHLAW_REPO_URL",
    "https://github.com/YOUR_USER/YOUR_REPO.git",
)
CLONE_DIR = Path("/content/phlaw-repo")


def find_backend(root: Path):
    """Monorepo: .../philippine-law-assistant/backend. Standalone: .../backend."""
    for rel in ("philippine-law-assistant/backend", "backend"):
        p = root / rel
        if (p / "app" / "prompts.py").is_file():
            return p
    return None


BACKEND_ROOT = find_backend(CLONE_DIR)
if BACKEND_ROOT is None and "YOUR_USER" not in REPO_URL:
    CLONE_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(CLONE_DIR)],
        check=False,
    )
    BACKEND_ROOT = find_backend(CLONE_DIR)

# --- Upload zip: uncomment and set ---
# BACKEND_ROOT = Path("/content/philippine-law-assistant/backend")

if BACKEND_ROOT is None:
    raise FileNotFoundError(
        "Set REPO_URL to your git URL, or upload philippine-law-assistant and set BACKEND_ROOT."
    )

BACKEND_ROOT = BACKEND_ROOT.resolve()
sys.path.insert(0, str(BACKEND_ROOT))

from app.prompts import PHILIPPINE_LAW_TUTOR_SYSTEM
from app.services.rag_simple import retrieve

print("Backend:", BACKEND_ROOT)

## 3) Load an instruct model on GPU

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = os.environ.get("PHLAW_MODEL_ID", "Qwen/Qwen2.5-1.5B-Instruct")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print("Loaded:", MODEL_ID)

## 4) Chat helper (RAG + generate)

In [ ]:
def build_context(docs):
    if not docs:
        return ""
    parts = ["Relevant provisions (cite these only):\n"]
    for r in docs:
        raw = str(r.get("content", ""))
        content = " ".join(raw.split())[:2000]
        parts.append(f"[{r['source']}]\n{content}\n")
    return "\n".join(parts)


def chat_colab(
    message: str,
    *,
    top_k: int = 5,
    max_new_tokens: int = 512,
    temperature: float = 0.6,
):
    docs = retrieve(message, top_k=top_k)
    ctx = build_context(docs)
    user = f"{ctx}\n\nUser question: {message}" if ctx else message
    messages = [
        {"role": "system", "content": PHILIPPINE_LAW_TUTOR_SYSTEM},
        {"role": "user", "content": user},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    gen_kw = {"max_new_tokens": max_new_tokens, "pad_token_id": tokenizer.eos_token_id}
    if temperature is not None and temperature > 0:
        gen_kw["do_sample"] = True
        gen_kw["temperature"] = temperature
    else:
        gen_kw["do_sample"] = False
    with torch.inference_mode():
        out = model.generate(**inputs, **gen_kw)
    gen = out[0, inputs["input_ids"].shape[1] :]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

In [ ]:
q = "What is due process under Article III of the 1987 Constitution?"
print(chat_colab(q))

## 5) Inspect RAG only (no LLM)

In [ ]:
for i, doc in enumerate(retrieve("writ of habeas corpus", top_k=3), 1):
    print(i, doc["source"], "—", doc["content"][:200].replace("\n", " "), "…")